# Conceptual Overview: Chamfer Distance for Pollen Embeddings

**Idea**: Each pollen sample is represented by more than one image view, and those views should be compared as a small set rather than as one fixed vector.

**Approach**: It uses embeddings from a previously trained representation model. For every pollen event, the notebook loads two embeddings that correspond to two orthoginal views of the same sample. These embeddings are then joined with the species labels so that the notebook can test whether similar embedding sets belong to the same pollen species.

Chamfer distance is used to compare two pollen samples. Conceptually, it asks: for each view of one sample, how close is the most similar view of the other sample? It does this in both directions and averages the result. This makes the comparison less sensitive to the order of the views and better suited to multi-view data.

The notebook uses a k-nearest-neighbor classifier. A test sample is classified by finding the most similar training samples under the Chamfer distance and assigning the species that receives the strongest neighbor vote. 

**Results**: The notebook evaluates how well this distance-based approach works. It reports overall accuracy, balanced accuracy, confusion matrices, and a classification report. These results show not only how often the classifier is correct overall, but also which pollen species are easy or difficult to distinguish from their learned representations.


Imports and paths

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [ ]:
def load_inference_results(filename: str) -> pd.DataFrame:
    data = np.load(filename)

    def make_repr(emb_key, proj_key, files_key):
        d = {
            "emb": data[emb_key].tolist(),
            "rec_path": data[files_key].tolist(),
        }
        if proj_key in data:
            d["proj"] = data[proj_key].tolist()
        return pd.DataFrame(d)

    repr1 = make_repr("emb1", "proj1", "files1")
    repr2 = make_repr("emb2", "proj2", "files2")

    df = pd.concat([repr1, repr2], ignore_index=True)
    labels_name = data.get("dataset")
    return df, labels_name

Load test labels

In [ ]:
test_labels = pd.read_csv("../data/final/poleno/isolated_test.csv")
# test_labels = pd.read_csv("../data/final/poleno/basic_test.csv")

Load embeddings

In [ ]:
print("Loading", repr["label"])
df_repr, _ = load_inference_results(repr["file"])

Pair views by sample

In [ ]:
df = pd.merge(df_repr, test_labels, on="rec_path", how='inner')
df = df.sort_values(["species", "event_id"]).reset_index(drop=True)

repr["emb1"] = np.vstack(df.loc[df["image_nr"]==0]["emb"].values)
repr["emb2"] = np.vstack(df.loc[df["image_nr"]==1]["emb"].values)
repr["proj1"] = np.vstack(df.loc[df["image_nr"]==0]["proj"].values)
repr["proj2"] = np.vstack(df.loc[df["image_nr"]==1]["proj"].values)
repr["species"] = df.loc[df["image_nr"]==0]["species"].values

Create train-test split

In [ ]:
emb_sets = np.stack([repr["emb1"], repr["emb2"]], axis=1)
species = np.asarray(repr["species"])

def capped_train_test_split(X, y, max_train_per_species, test_size=0.2, random_state=5):
    rng = np.random.default_rng(random_state)

    train_indices = []
    test_indices = []

    for sp in np.unique(y):
        sp_indices = np.flatnonzero(y == sp)
        rng.shuffle(sp_indices)

        # Normal stratified split size for this species
        n_train = int(round(len(sp_indices) * (1 - test_size)))

        # Cap train samples for this species
        n_train = min(n_train, max_train_per_species)

        train_indices.extend(sp_indices[:n_train])
        test_indices.extend(sp_indices[n_train:])

    train_indices = np.asarray(train_indices)
    test_indices = np.asarray(test_indices)

    # Shuffle final splits
    rng.shuffle(train_indices)
    rng.shuffle(test_indices)

    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]


X_train, X_test, y_train, y_test = capped_train_test_split(
    emb_sets,
    species,
    max_train_per_species=320,  
    test_size=0.2,
    random_state=42,
)

Define Chamfer kNN

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F


def _get_device(device=None):
    """
    Resolve the torch device.

    If device is None, use CUDA when available, otherwise CPU.
    """
    if device is None:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device)


def _ensure_3d(X, device=None, dtype=torch.float32):
    """
    Ensures X has shape:
        (n_samples, n_views, embedding_dim)

    If X is 2D, interprets it as:
        (n_samples, n_views, 1)
    """
    device = _get_device(device)

    if not torch.is_tensor(X):
        X = torch.as_tensor(X, dtype=dtype, device=device)
    else:
        X = X.to(device=device, dtype=dtype)

    if X.ndim == 2:
        X = X.unsqueeze(-1)
    elif X.ndim != 3:
        raise ValueError(
            "X must have shape (n_samples, n_views) or "
            "(n_samples, n_views, embedding_dim)."
        )

    return X


def l2_normalize(X, eps=1e-12, device=None):
    """
    L2-normalize vectors along the embedding dimension.
    """
    X = _ensure_3d(X, device=device)
    return F.normalize(X, p=2, dim=-1, eps=eps)


def pairwise_view_distances(A, B, metric="cosine", eps=1e-12):
    """
    Computes all pairwise view distances between samples in A and B.

    A shape: (n_a, n_views_a, dim)
    B shape: (n_b, n_views_b, dim)

    Returns:
        D shape: (n_a, n_b, n_views_a, n_views_b)
    """
    if not torch.is_tensor(A) or not torch.is_tensor(B):
        raise TypeError("A and B must be torch tensors. Use _ensure_3d first.")

    if A.ndim != 3 or B.ndim != 3:
        raise ValueError("A and B must both have shape (n_samples, n_views, dim).")

    if A.shape[-1] != B.shape[-1]:
        raise ValueError("A and B must have the same embedding dimension.")

    if A.device != B.device:
        B = B.to(A.device)

    if metric == "sqeuclidean":
        # ||a - b||^2 = ||a||^2 + ||b||^2 - 2 a·b
        A2 = (A * A).sum(dim=-1)  # (n_a, n_views_a)
        B2 = (B * B).sum(dim=-1)  # (n_b, n_views_b)

        cross = torch.einsum("avd,bwd->abvw", A, B)
        D = A2[:, None, :, None] + B2[None, :, None, :] - 2.0 * cross
        return torch.clamp(D, min=0.0)

    if metric == "euclidean":
        D2 = pairwise_view_distances(A, B, metric="sqeuclidean", eps=eps)
        return torch.sqrt(torch.clamp(D2, min=0.0))

    if metric == "cosine":
        A_norm = F.normalize(A, p=2, dim=-1, eps=eps)
        B_norm = F.normalize(B, p=2, dim=-1, eps=eps)
        sim = torch.einsum("avd,bwd->abvw", A_norm, B_norm)
        return 1.0 - sim

    raise ValueError("metric must be one of: 'cosine', 'euclidean', 'sqeuclidean'.")


def chamfer_distance_matrix(A, B, metric="cosine", reduction="mean"):
    """
    Computes symmetric Chamfer distances between all samples in A and B.

    A shape: (n_a, n_views_a, dim)
    B shape: (n_b, n_views_b, dim)

    Returns:
        C shape: (n_a, n_b)

    Symmetric Chamfer:

        C(A, B) =
            mean_a min_b d(a, b)
            +
            mean_b min_a d(a, b)

    If reduction='mean', divides the final sum by 2.
    If reduction='sum', keeps the raw two-direction sum.
    """
    D = pairwise_view_distances(A, B, metric=metric)
    # D shape: (n_a, n_b, n_views_a, n_views_b)

    a_to_b = D.min(dim=3).values.mean(dim=2)  # (n_a, n_b)
    b_to_a = D.min(dim=2).values.mean(dim=2)  # (n_a, n_b)

    C = a_to_b + b_to_a

    if reduction == "mean":
        C = 0.5 * C
    elif reduction == "sum":
        pass
    else:
        raise ValueError("reduction must be 'mean' or 'sum'.")

    return C


class ChamferKNNClassifier:
    """
    kNN classifier for unordered sets of view embeddings using Chamfer distance.

    Parameters
    ----------
    n_neighbors : int
        Number of nearest reference samples to vote over.

    metric : {'cosine', 'euclidean', 'sqeuclidean'}
        Distance used between individual view embeddings.

    weights : {'uniform', 'distance'}
        Voting strategy.

    normalize : bool
        Whether to L2-normalize input view vectors before storing/comparing.
        Usually recommended for embedding vectors.

    batch_size : int or None
        If set, query samples are processed in batches to avoid huge memory use.

    device : str, torch.device, or None
        Device used for distance computation. If None, uses CUDA when available.
    """

    def __init__(
        self,
        n_neighbors=3,
        metric="cosine",
        weights="distance",
        normalize=True,
        batch_size=None,
        device=None,
        dtype=torch.float32,
    ):
        self.n_neighbors = n_neighbors
        self.metric = metric
        self.weights = weights
        self.normalize = normalize
        self.batch_size = batch_size
        self.device = _get_device(device)
        self.dtype = dtype

    def fit(self, X, y):
        X = _ensure_3d(X, device=self.device, dtype=self.dtype)

        if self.normalize:
            X = l2_normalize(X, device=self.device)

        # Keep labels on CPU as a NumPy array so string/object labels work.
        # Distance computation still happens on CUDA/torch.
        y = np.asarray(y)

        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of samples.")

        self.X_train_ = X
        self.y_train_ = y
        self.classes_ = np.unique(y)
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}

        return self

    @torch.no_grad()
    def kneighbors(self, X, return_distance=True):
        if not hasattr(self, "X_train_"):
            raise RuntimeError("This ChamferKNNClassifier instance is not fitted yet.")

        X = _ensure_3d(X, device=self.device, dtype=self.dtype)

        if self.normalize:
            X = l2_normalize(X, device=self.device)

        n_query = len(X)
        k = min(self.n_neighbors, len(self.X_train_))

        all_indices = []
        all_distances = []

        if self.batch_size is None:
            batches = [(0, n_query)]
        else:
            batches = [
                (start, min(start + self.batch_size, n_query))
                for start in range(0, n_query, self.batch_size)
            ]

        for start, end in batches:
            D = chamfer_distance_matrix(
                X[start:end],
                self.X_train_,
                metric=self.metric,
                reduction="mean",
            )

            # torch.topk with largest=False gives the k nearest neighbors.
            dist_sorted, idx_sorted = torch.topk(D, k=k, dim=1, largest=False, sorted=True)

            all_indices.append(idx_sorted)
            all_distances.append(dist_sorted)

        indices = torch.cat(all_indices, dim=0)
        distances = torch.cat(all_distances, dim=0)

        if return_distance:
            return distances, indices
        return indices

    @torch.no_grad()
    def predict(self, X):
        distances, indices = self.kneighbors(X, return_distance=True)

        # Move only the small k-neighbor result back to CPU for label voting.
        indices_np = indices.detach().cpu().numpy()
        distances_np = distances.detach().cpu().numpy()
        neighbor_labels = self.y_train_[indices_np]

        preds = []

        for labs, dists in zip(neighbor_labels, distances_np):
            if self.weights == "uniform":
                scores = {}
                for label in labs:
                    scores[label] = scores.get(label, 0.0) + 1.0
                pred = max(scores, key=scores.get)

            elif self.weights == "distance":
                scores = {}
                for label, dist in zip(labs, dists):
                    weight = 1.0 / (float(dist) + 1e-12)
                    scores[label] = scores.get(label, 0.0) + weight
                pred = max(scores, key=scores.get)

            else:
                raise ValueError("weights must be 'uniform' or 'distance'.")

            preds.append(pred)

        return np.asarray(preds)

    @torch.no_grad()
    def predict_proba(self, X):
        distances, indices = self.kneighbors(X, return_distance=True)

        # Move only the small k-neighbor result back to CPU for label aggregation.
        indices_np = indices.detach().cpu().numpy()
        distances_np = distances.detach().cpu().numpy()
        neighbor_labels = self.y_train_[indices_np]

        proba = np.zeros((len(neighbor_labels), len(self.classes_)), dtype=np.float32)

        for i, (labs, dists) in enumerate(zip(neighbor_labels, distances_np)):
            if self.weights == "uniform":
                weights = np.ones_like(dists, dtype=np.float32)
            elif self.weights == "distance":
                weights = 1.0 / (dists + 1e-12)
            else:
                raise ValueError("weights must be 'uniform' or 'distance'.")

            for label, weight in zip(labs, weights):
                proba[i, self.class_to_idx_[label]] += weight

            total = proba[i].sum()
            if total > 0:
                proba[i] /= total

        return proba


Fit and predict

In [ ]:
clf = ChamferKNNClassifier(
    n_neighbors=10,
    metric="cosine",
    weights="distance",
    normalize=True,
    batch_size=32,
    device=None,  # auto: cuda if available, else cpu
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)

print("Device:", clf.device)
print("Predictions:", y_pred)
print("Probabilities shape:", y_probs.shape)

Show classifier classes

In [ ]:
print("Classes", clf.classes_)

Show results

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Overall accuracy
acc = accuracy_score(y_test, y_pred)

# Class-balanced accuracy = mean recall across species
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Overall accuracy:        {acc:.4f}")
print(f"Balanced accuracy:       {bal_acc:.4f}")

In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=clf.classes_
)

fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, xticks_rotation=90, cmap="Blues", values_format="d")

plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Plot Normalized Matrix
cm_norm = confusion_matrix(
    y_test,
    y_pred,
    labels=clf.classes_,
    normalize="true"
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=clf.classes_
)

fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, xticks_rotation=90, cmap="Blues", values_format=".2f")

plt.title("Normalized Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    labels=clf.classes_,
    zero_division=0
))